In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from typing import Any

from project_root import PROJECT_ROOT,DATASETS_ROOT

import fiftyone as fo
import fiftyone.utils.torch as fout


import torchvision as tv

from scripts.model_serialization import load_model

no_grad_guard = torch.no_grad()

2025-04-22 13:54:43.020241: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-22 13:54:43.027081: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745322883.034946   44242 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745322883.037351   44242 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745322883.043499   44242 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [3]:
# dataset_name =         "zoo-elephants-identity-tracks"
# ds =fo.load_dataset(dataset_name)


def make_ds():
    # return fo.Dataset.from_dir(
    #     dataset_type=fo.types.ImageDirectory,
    #     dataset_dir="/media/dherrera/ElephantExternal/elephants/improve/detection/2025-01-25",
    #     persistent=False,
    # )
    return fo.Dataset.from_dir(
        dataset_type=fo.types.ImageClassificationDirectoryTree,
        dataset_dir="/home/dherrera/data/elephants/behaviour/sleep_v2",
        persistent=False,
    )


ds = make_ds()
# classes = sorted(ds.classes["ground_truth"])
# print(classes)
print(ds)

 100% |███████████████| 4541/4541 [810.4ms elapsed, 0s remaining, 5.6K samples/s]      
Name:        2025.04.22.13.55.09
Media type:  image
Num samples: 4541
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)


In [4]:
# Compute uniqueness
import fiftyone.brain as fob

session = fo.launch_app(ds, auto=False)
session.open_tab()

Session launched. Run `session.show()` to open the App in a cell output.


<IPython.core.display.Javascript object>

In [5]:
from tqdm.auto import tqdm

dup_result = fob.compute_near_duplicates(ds)
print(len(dup_result.duplicate_ids))
session.view = dup_result.duplicates_view()

DELETE_DUPLICATES = False
if DELETE_DUPLICATES:
    while len(dup_result.duplicate_ids) > 0:
        print(f"Deleting {len(dup_result.duplicate_ids)} duplicate files")
        for id in tqdm(dup_result.duplicate_ids):
            filepath = ds[id]["filepath"]
            Path.unlink(filepath)
        ds = make_ds()
        dup_result = fob.compute_near_duplicates(ds)
        print(len(dup_result.duplicate_ids))

Computing embeddings...
 100% |███████████████| 1810/1810 [34.3s elapsed, 0s remaining, 58.0 samples/s]      
Computing duplicate samples...
Generating index for 1810 embeddings...
Index complete
Generating neighbors graph for 243 embeddings...
Index complete
Duplicates computation complete
1567


In [14]:
from fiftyone import ViewField as F

THRESHOLD = 0.01
fob.compute_uniqueness(ds)
# session.view = ds.sort_by("uniqueness", reverse=True)

ds_bad = ds.match(F("uniqueness") < THRESHOLD)
print(f"Samples to drop: {len(ds_bad)}")
session.view = ds_bad

from tqdm.auto import tqdm

DELETE_BAD = False
if DELETE_BAD:
    while len(ds_bad) > 0:
        print(f"Deleting {len(ds_bad)} low-uniqueness files")
        for sample in tqdm(ds_bad):
            filepath = sample["filepath"]
            Path.unlink(filepath)
        ds = make_ds()
        fob.compute_uniqueness(ds)
        ds_bad = ds.match(F("uniqueness") < THRESHOLD)

session.view = ds_bad

Computing embeddings...
 100% |███████████████| 4541/4541 [958.4ms elapsed, 0s remaining, 4.7K samples/s]      
Computing uniqueness...
Generating index for 4541 embeddings...
Index complete
Uniqueness computation complete
Samples to drop: 2588


In [13]:
session.view = ds.match(F("uniqueness") >= 0.01)